# Chapter 4 — Results Figures

Generates the figures used in Chapter 4: study area context, before/after imagery, ground truth, model predictions, and a fusion-versus-optical agreement analysis. Each figure is saved individually for composition into the thesis document.

**Project:** Development and Evaluation of a Deep Learning Model for Flood Detection and Drought Prediction Using Satellite Remote Sensing Data in South Africa
**Study Area:** KwaZulu-Natal, South Africa
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Reference Product:** UNOSAT FL20220418ZAF
**Student:** Athindothe Valencia Marubini
**Student No:** 219160643
**Supervisor:** Prof. IE Davidson
**Co-Supervisor:** Dr O.P Babalola
**Institution:** Cape Peninsula University of Technology (CPUT)

---
## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install rasterio torch --quiet

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds, bounds as window_bounds
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

ROOT       = '/content/drive/MyDrive/KZN_Research_Colab/'
STACK_PATH = ROOT + 'Stacked_4tile/flood_stack_4tile_v1.tif'
LABEL_PATH = ROOT + 'Mosaic_4tile/flood_label_mosaic.tif'
S2_DIR     = ROOT + 'Mosaic_4tile/S2_bands/'
MODEL_DIR  = ROOT + 'Models_4tile/'
FIG_DIR    = ROOT + 'Chapter4_Figures_Final/'
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATE_S2_PRE  = '29 March 2022'
DATE_S2_POST = '28 April 2022'
DATE_S1_PRE  = '24 March 2022'
DATE_S1_POST = '17 April 2022'

COLOR_FLOOD     = (0x4A/255, 0x14/255, 0x86/255)   # deep violet-purple
COLOR_AGREE     = (0xFF/255, 0x00/255, 0x9F/255)   # bright magenta
COLOR_MISS      = (0xFF/255, 0xE6/255, 0x00/255)   # bright yellow
COLOR_DISAGREE  = (0x00/255, 0xD4/255, 0xFF/255)   # bright cyan

print('Setup complete. Device:', device)

---
## Study Area Extent

In [ ]:
with rasterio.open(LABEL_PATH) as src:
    label_full = src.read(1)
    label_transform = src.transform

rows_f, cols_f = np.where(label_full == 1)
buffer_px = 150
row_min = max(0, rows_f.min() - buffer_px)
row_max = min(label_full.shape[0], rows_f.max() + buffer_px)
col_min = max(0, cols_f.min() - buffer_px)
col_max = min(label_full.shape[1], cols_f.max() + buffer_px)

ZOOM_WINDOW = Window(col_min, row_min, col_max - col_min, row_max - row_min)
ZOOM_BOUNDS = window_bounds(ZOOM_WINDOW, label_transform)

def get_window(raster_path, geo_bounds=ZOOM_BOUNDS):
    with rasterio.open(raster_path) as src:
        return from_bounds(*geo_bounds, transform=src.transform)

del label_full
print('Study area extent defined.')

---
## Reading and Plotting Functions

In [ ]:
def read_rgb(period, geo_bounds=ZOOM_BOUNDS):
    paths = {
        'B04': S2_DIR + f'S2_B04_{period}.tif',
        'B03': S2_DIR + f'S2_B03_{period}.tif',
        'B02': S2_DIR + f'S2_B02_{period}.tif',
    }
    bands = {}
    for name, path in paths.items():
        with rasterio.open(path) as src:
            w = from_bounds(*geo_bounds, transform=src.transform)
            bands[name] = src.read(1, window=w)
    rgb = np.stack([bands['B04'], bands['B03'], bands['B02']], axis=-1).astype(np.float32)
    valid = rgb[rgb > 0]
    p2, p98 = np.percentile(valid, (2, 98)) if valid.size > 0 else (0, 1)
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    return rgb

def read_rgb_overview(period, target_width=2000):
    paths = {
        'B04': S2_DIR + f'S2_B04_{period}.tif',
        'B03': S2_DIR + f'S2_B03_{period}.tif',
        'B02': S2_DIR + f'S2_B02_{period}.tif',
    }
    with rasterio.open(paths['B04']) as src:
        full_w, full_h = src.width, src.height
    scale = target_width / full_w
    out_w, out_h = int(full_w * scale), int(full_h * scale)

    bands = {}
    for name, path in paths.items():
        with rasterio.open(path) as src:
            bands[name] = src.read(1, out_shape=(out_h, out_w), resampling=rasterio.enums.Resampling.average)
    rgb = np.stack([bands['B04'], bands['B03'], bands['B02']], axis=-1).astype(np.float32)
    valid = rgb[rgb > 0]
    p2, p98 = np.percentile(valid, (2, 98)) if valid.size > 0 else (0, 1)
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    return rgb

def read_band_geo(raster_path, band_idx=1, geo_bounds=ZOOM_BOUNDS):
    with rasterio.open(raster_path) as src:
        w = from_bounds(*geo_bounds, transform=src.transform)
        return src.read(band_idx, window=w)

def crop_to_common(*arrays):
    h = min(a.shape[0] for a in arrays)
    w = min(a.shape[1] for a in arrays)
    return [a[:h, :w] for a in arrays]

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR + filename, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Saved: {filename}')

def plot_rgb(rgb, title, filename, figsize=(10, 10)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    save_figure(fig, filename)

def plot_sar(data, title, filename, vmin=-25, vmax=5, figsize=(10, 10)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(data, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    save_figure(fig, filename)

def plot_overlay(rgb_bg, mask, color, title, filename, alpha=0.7, figsize=(10, 10)):
    rgb_c, mask_c = crop_to_common(rgb_bg, mask)
    overlay = np.zeros((*mask_c.shape, 4), dtype=np.float32)
    overlay[..., 0], overlay[..., 1], overlay[..., 2] = color
    overlay[..., 3] = np.where(mask_c == 1, alpha, 0.0)

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb_c)
    ax.imshow(overlay)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    save_figure(fig, filename)

print('Functions ready.')

---
## Figure 1-2: Study Area Overview

In [ ]:
rgb_overview_pre = read_rgb_overview('pre')
plot_rgb(rgb_overview_pre, f'Study Area Overview — Pre-Flood\n{DATE_S2_PRE}',
         '01_overview_pre.png')
del rgb_overview_pre

In [ ]:
rgb_overview_post = read_rgb_overview('post')
plot_rgb(rgb_overview_post, f'Study Area Overview — Post-Flood\n{DATE_S2_POST}',
         '02_overview_post.png')
del rgb_overview_post

---
## Figure 3-4: Optical Imagery, Before and After

In [ ]:
rgb_zoom_pre = read_rgb('pre')
plot_rgb(rgb_zoom_pre, f'Optical Imagery — Pre-Flood\n{DATE_S2_PRE}',
         '03_optical_pre.png')

In [ ]:
rgb_zoom_post = read_rgb('post')
plot_rgb(rgb_zoom_post, f'Optical Imagery — Post-Flood\n{DATE_S2_POST}',
         '04_optical_post.png')

---
## Figure 5-6: SAR Imagery, Before and After

In [ ]:
vv_pre_zoom  = read_band_geo(STACK_PATH, band_idx=10)
vv_post_zoom = read_band_geo(STACK_PATH, band_idx=11)

plot_sar(vv_pre_zoom, f'SAR Imagery (VV) — Pre-Flood\n{DATE_S1_PRE}', '05_sar_pre.png')
plot_sar(vv_post_zoom, f'SAR Imagery (VV) — Post-Flood\n{DATE_S1_POST}', '06_sar_post.png')

---
## Load Trained Models

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.enc4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base*8, base*16)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out_conv = nn.Conv2d(base, out_channels, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.up4(b);  d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4); d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3); d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2); d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out_conv(d1)

fusion_model = UNet(in_channels=12).to(device)
fusion_model.load_state_dict(torch.load(MODEL_DIR + 'fusion_unet_4tile_best.pt', map_location=device))
fusion_model.eval()

optical_model = UNet(in_channels=9).to(device)
optical_model.load_state_dict(torch.load(MODEL_DIR + 'optical_unet_4tile_best.pt', map_location=device))
optical_model.eval()

print('Models loaded.')

---
## Run Model Inference

In [ ]:
OPTICAL_CHANNELS = list(range(9))
PATCH = 128

def run_inference_tiled(model, channel_subset=None, geo_bounds=ZOOM_BOUNDS):
    with rasterio.open(STACK_PATH) as src:
        window = from_bounds(*geo_bounds, transform=src.transform)
    zoom_h, zoom_w = int(round(window.height)), int(round(window.width))
    pred_full = np.zeros((zoom_h, zoom_w), dtype=np.float32)

    with rasterio.open(STACK_PATH) as src:
        for row in range(0, zoom_h, PATCH):
            for col in range(0, zoom_w, PATCH):
                h = min(PATCH, zoom_h - row)
                w = min(PATCH, zoom_w - col)
                tile_window = Window(window.col_off + col, window.row_off + row, w, h)
                X = src.read(window=tile_window).astype(np.float32)
                if channel_subset is not None:
                    X = X[channel_subset]
                X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

                if X.shape[1] < PATCH or X.shape[2] < PATCH:
                    pad_h, pad_w = PATCH - X.shape[1], PATCH - X.shape[2]
                    X = np.pad(X, ((0,0),(0,pad_h),(0,pad_w)), mode='constant')

                X_tensor = torch.from_numpy(X).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = model(X_tensor)
                    probs = torch.sigmoid(logits).cpu().numpy()[0, 0]
                pred_full[row:row+h, col:col+w] = probs[:h, :w]
    return pred_full

fusion_pred = run_inference_tiled(fusion_model, channel_subset=None)
fusion_binary = (fusion_pred > 0.5).astype(np.uint8)

optical_pred = run_inference_tiled(optical_model, channel_subset=OPTICAL_CHANNELS)
optical_binary = (optical_pred > 0.5).astype(np.uint8)

print('Inference complete.')

---
## Figure 7: Ground Truth

In [ ]:
label_zoom = read_band_geo(LABEL_PATH, band_idx=1)

plot_overlay(
    rgb_zoom_post, label_zoom, COLOR_FLOOD,
    f'UNOSAT Ground Truth — Flood Extent\nBackground: Post-Flood Optical, {DATE_S2_POST}',
    '07_ground_truth.png'
)

---
## Figure 8-9: Model Predictions

In [ ]:
plot_overlay(
    rgb_zoom_post, fusion_binary, COLOR_FLOOD,
    f'Fusion Model Prediction (12-Channel)\nBackground: Post-Flood Optical, {DATE_S2_POST}',
    '08_prediction_fusion.png'
)

In [ ]:
plot_overlay(
    rgb_zoom_post, optical_binary, COLOR_FLOOD,
    f'Optical Model Prediction (9-Channel)\nBackground: Post-Flood Optical, {DATE_S2_POST}',
    '09_prediction_optical.png'
)

---
## Figure 10: Agreement Analysis

In [ ]:
label_c, fusion_c, optical_c = crop_to_common(label_zoom, fusion_binary, optical_binary)
rgb_c = crop_to_common(rgb_zoom_post, label_c)[0]
label_c, fusion_c, optical_c = crop_to_common(label_c, fusion_c, optical_c)

label_bool   = (label_c == 1)
fusion_bool  = (fusion_c == 1)
optical_bool = (optical_c == 1)

both_correct    = (fusion_bool == label_bool) & (optical_bool == label_bool) & label_bool
both_missed     = label_bool & ~fusion_bool & ~optical_bool
models_disagree = (fusion_bool != optical_bool)

overlay = np.zeros((*label_bool.shape, 4), dtype=np.float32)
def set_overlay(mask, color, alpha=0.75):
    overlay[mask, 0], overlay[mask, 1], overlay[mask, 2] = color
    overlay[mask, 3] = alpha

set_overlay(both_correct, COLOR_AGREE)
set_overlay(both_missed, COLOR_MISS)
set_overlay(models_disagree, COLOR_DISAGREE)

fig, ax = plt.subplots(figsize=(11, 10))
ax.imshow(rgb_c)
ax.imshow(overlay)
ax.set_title(f'Fusion vs Optical — Agreement Analysis\nBackground: Post-Flood Optical, {DATE_S2_POST}', fontsize=14)
ax.axis('off')

legend_patches = [
    mpatches.Patch(color=COLOR_AGREE, label='Both models correct (matches ground truth)'),
    mpatches.Patch(color=COLOR_MISS, label='Both models missed (ground truth flood, neither detected)'),
    mpatches.Patch(color=COLOR_DISAGREE, label='Models disagree with each other'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9, framealpha=0.9)

save_figure(fig, '10_agreement_analysis.png')

total = label_bool.size
print(f'Both correct: {100*both_correct.sum()/total:.2f}%')
print(f'Both missed : {100*both_missed.sum()/total:.2f}%')
print(f'Disagree    : {100*models_disagree.sum()/total:.2f}%')

---
## Summary

In [ ]:
print('All figures saved to:', FIG_DIR)
print()
for f in sorted(os.listdir(FIG_DIR)):
    print(' ', f)

---
## Multi-Zone Detailed Figures — Harbour and uMngeni River Mouth


In [ ]:
COLOR_FLOOD_ZONE = (0xFF/255, 0x00/255, 0x9F/255)   # magenta - flood/water, all zone figures
COLOR_MISSED_ZONE = (0xE0/255, 0x10/255, 0x10/255)   # red - missed flood (false negative)
COLOR_DISAGREE_ZONE = (0x34/255, 0xE3/255, 0xFE/255) # #34E3FE bright cyan - model disagreement
COLOR_FP_ZONE = (0xFF/255, 0xE6/255, 0x00/255)       # #FFE600 yellow - false positive

with rasterio.open(LABEL_PATH) as src:
    full_label = src.read(1)
    full_transform = src.transform

rows_f, cols_f = np.where(full_label == 1)


from sklearn.cluster import DBSCAN

coords = np.column_stack([rows_f, cols_f])
clustering = DBSCAN(eps=50, min_samples=10).fit(coords)
labels = clustering.labels_
unique_clusters = sorted(set(labels) - {-1})  # exclude noise (-1)

print(f'Found {len(unique_clusters)} distinct flood clusters:')
cluster_info = []
for c in unique_clusters:
    mask = labels == c
    n_px = mask.sum()
    row_c = int(np.median(rows_f[mask]))
    col_c = int(np.median(cols_f[mask]))
    cluster_info.append((c, n_px, row_c, col_c))
    print(f'  Cluster {c}: {n_px} pixels, centre (row={row_c}, col={col_c})')


cluster_info.sort(key=lambda x: -x[1])  # sort by size, descending
harbour_cluster = cluster_info[0]
harbour_row_c, harbour_col_c = harbour_cluster[2], harbour_cluster[3]

cluster_info.sort(key=lambda x: x[2])  # sort by row (north to south)
north_cluster = cluster_info[0]
umngeni_row_c, umngeni_col_c = north_cluster[2], north_cluster[3]

print(f'\nHarbour zone   -> cluster {harbour_cluster[0]} ({harbour_cluster[1]} px), centre ({harbour_row_c}, {harbour_col_c})')
print(f'uMngeni zone   -> cluster {north_cluster[0]} ({north_cluster[1]} px), centre ({umngeni_row_c}, {umngeni_col_c})')

if harbour_cluster[0] == north_cluster[0]:
    print('\nNOTE: largest and northernmost cluster are the SAME cluster.')


ZONE_HALF_SIZE = 600

def zone_bounds(row_c, col_c, half=ZONE_HALF_SIZE):
    w = Window(col_c - half, row_c - half, half*2, half*2)
    return window_bounds(w, full_transform)

HARBOUR_BOUNDS = zone_bounds(harbour_row_c, harbour_col_c)
UMNGENI_BOUNDS = zone_bounds(umngeni_row_c, umngeni_col_c)

del full_label

In [ ]:
def generate_zone_figures(zone_name, zone_bounds):
    print(f'=== {zone_name} ===\n')

    rgb_pre  = read_rgb('pre',  geo_bounds=zone_bounds)
    rgb_post = read_rgb('post', geo_bounds=zone_bounds)
    label_z  = read_band_geo(LABEL_PATH, band_idx=1, geo_bounds=zone_bounds)

    fusion_pred_z  = run_inference_tiled(fusion_model,  channel_subset=None,            geo_bounds=zone_bounds)
    optical_pred_z = run_inference_tiled(optical_model, channel_subset=OPTICAL_CHANNELS, geo_bounds=zone_bounds)
    fusion_bin_z  = (fusion_pred_z  > 0.5).astype(np.uint8)
    optical_bin_z = (optical_pred_z > 0.5).astype(np.uint8)

    slug = zone_name.lower().replace(' ', '_')

    plot_rgb(rgb_pre,  f'{zone_name} — Pre-Flood Optical\n{DATE_S2_PRE}',  f'{slug}_01_pre.png')
    plot_rgb(rgb_post, f'{zone_name} — Post-Flood Optical\n{DATE_S2_POST}', f'{slug}_02_post.png')

    plot_overlay(rgb_post, label_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — UNOSAT Ground Truth\nBackground: {DATE_S2_POST}',
                 f'{slug}_03_ground_truth.png')

    plot_overlay(rgb_post, fusion_bin_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — Fusion Model Prediction\nBackground: {DATE_S2_POST}',
                 f'{slug}_04_fusion.png')

    plot_overlay(rgb_post, optical_bin_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — Optical Model Prediction\nBackground: {DATE_S2_POST}',
                 f'{slug}_05_optical.png')

    # Error analysis specific to this zone,
    label_c, fusion_c = crop_to_common(label_z, fusion_bin_z)
    rgb_c = crop_to_common(rgb_post, label_c)[0]
    label_c, fusion_c = crop_to_common(label_c, fusion_c)
    label_bool = (label_c == 1)
    fusion_bool = (fusion_c == 1)

    correct = label_bool & fusion_bool
    missed  = label_bool & ~fusion_bool
    false_pos = ~label_bool & fusion_bool

    overlay = np.zeros((*label_bool.shape, 4), dtype=np.float32)
    def set_ov(mask, color, alpha=0.75):
        overlay[mask, 0], overlay[mask, 1], overlay[mask, 2] = color
        overlay[mask, 3] = alpha
    set_ov(correct, COLOR_FLOOD_ZONE)
    set_ov(missed, COLOR_MISSED_ZONE)
    set_ov(false_pos, COLOR_FP_ZONE)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(rgb_c)
    ax.imshow(overlay)
    ax.set_title(f'{zone_name} — Fusion Model Error Analysis\nBackground: {DATE_S2_POST}', fontsize=14)
    ax.axis('off')
    legend_patches = [
        mpatches.Patch(color=COLOR_FLOOD_ZONE, label='Correctly detected flood'),
        mpatches.Patch(color=COLOR_MISSED_ZONE, label='Missed (false negative)'),
        mpatches.Patch(color=COLOR_FP_ZONE, label='False positive'),
    ]
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9, framealpha=0.9)
    save_figure(fig, f'{slug}_06_error_analysis.png')

    print()
    return rgb_pre, rgb_post, label_z, fusion_bin_z, optical_bin_z

_ = generate_zone_figures('Harbour Area', HARBOUR_BOUNDS)

In [ ]:
_ = generate_zone_figures('uMngeni River Mouth', UMNGENI_BOUNDS)

In [ ]:


already_done = {harbour_cluster[0], north_cluster[0]}
remaining_clusters = [c for c in cluster_info if c[0] not in already_done and c[1] >= 10]

print(f'{len(remaining_clusters)} additional cluster(s) to process:\n')
for c in remaining_clusters:
    print(f'  Cluster {c[0]}: {c[1]} pixels, centre (row={c[2]}, col={c[3]})')
print()

for cluster_id, n_px, row_c, col_c in remaining_clusters:
    zone_name = f'Flood Area {cluster_id}'
    bounds = zone_bounds(row_c, col_c)
    _ = generate_zone_figures(zone_name, bounds)

In [ ]:
for cluster_id, n_px, row_c, col_c in remaining_clusters:
    zone_name = 'uMngeni River Corridor'
    bounds = zone_bounds(row_c, col_c)

    print(f'=== {zone_name} ===\n')

    rgb_pre  = read_rgb('pre',  geo_bounds=bounds)
    rgb_post = read_rgb('post', geo_bounds=bounds)
    label_z  = read_band_geo(LABEL_PATH, band_idx=1, geo_bounds=bounds)

    fusion_pred_z  = run_inference_tiled(fusion_model,  channel_subset=None,            geo_bounds=bounds)
    optical_pred_z = run_inference_tiled(optical_model, channel_subset=OPTICAL_CHANNELS, geo_bounds=bounds)
    fusion_bin_z  = (fusion_pred_z  > 0.5).astype(np.uint8)
    optical_bin_z = (optical_pred_z > 0.5).astype(np.uint8)

    slug = f'flood_area_{cluster_id}'

    plot_rgb(rgb_pre,  f'{zone_name} — Pre-Flood Optical\n{DATE_S2_PRE}',   f'{slug}_01_pre.png')
    plot_rgb(rgb_post, f'{zone_name} — Post-Flood Optical\n{DATE_S2_POST}',  f'{slug}_02_post.png')
    plot_overlay(rgb_post, label_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — UNOSAT Ground Truth\nBackground: {DATE_S2_POST}',
                 f'{slug}_03_ground_truth.png')
    plot_overlay(rgb_post, fusion_bin_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — Fusion Model Prediction\nBackground: {DATE_S2_POST}',
                 f'{slug}_04_fusion.png')
    plot_overlay(rgb_post, optical_bin_z, COLOR_FLOOD_ZONE,
                 f'{zone_name} — Optical Model Prediction\nBackground: {DATE_S2_POST}',
                 f'{slug}_05_optical.png')

    label_c, fusion_c = crop_to_common(label_z, fusion_bin_z)
    rgb_c = crop_to_common(rgb_post, label_c)[0]
    label_c, fusion_c = crop_to_common(label_c, fusion_c)
    label_bool  = (label_c == 1)
    fusion_bool = (fusion_c == 1)
    correct   = label_bool & fusion_bool
    missed    = label_bool & ~fusion_bool
    false_pos = ~label_bool & fusion_bool

    overlay = np.zeros((*label_bool.shape, 4), dtype=np.float32)
    def set_ov(mask, color, alpha=0.75):
        overlay[mask, 0], overlay[mask, 1], overlay[mask, 2] = color
        overlay[mask, 3] = alpha
    set_ov(correct,   COLOR_FLOOD_ZONE)
    set_ov(missed,    COLOR_MISSED_ZONE)
    set_ov(false_pos, COLOR_FP_ZONE)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(rgb_c)
    ax.imshow(overlay)
    ax.set_title(f'{zone_name} — Fusion Model Error Analysis\nBackground: {DATE_S2_POST}', fontsize=14)
    ax.axis('off')
    legend_patches = [
        mpatches.Patch(color=COLOR_FLOOD_ZONE,  label='Correctly detected flood'),
        mpatches.Patch(color=COLOR_MISSED_ZONE, label='Missed (false negative)'),
        mpatches.Patch(color=COLOR_FP_ZONE,     label='False positive'),
    ]
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9, framealpha=0.9)
    save_figure(fig, f'{slug}_06_error_analysis.png')
    print()